In [32]:
import math
import torch
from torch import nn

import myattentionlib as mal

In [33]:
class MultiHeadAttention(nn.Module):
    """多头注意力 \n
        此处设定p_q = p_k = p_v = p_o / h, 以防计算代价和参数代价增长"""
    # p_q = p_k = p_v = num_hiddens

    def __init__(self, key_size, query_size, value_size, num_hiddens, num_heads, dropout, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.attention = mal.DotProductAttention(dropout)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=bias)
        self.W_k = nn.Linear(key_size, num_hiddens, bias=bias)
        self.W_v = nn.Linear(value_size, num_hiddens, bias=bias)      
        # 这里分开考虑W_o作用在一个head上 
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens):
        """ 
        Inputs: 
        queries, keys, values:  (batch_size, no. of queries or k-vs, 
                                query/key/value _size)
        valid_lens: (batch_size, ) or (batch_size, no. of queries)

        Outputs:
        self.W_o(output_concat):    (batch_size, no. of queries, num_hiddens)
        """

        # LHS queries, keys, values: 
        # (batch_size * num_heads, no. of queries or k-vs, 
        # num_hiddens/num_heads)
        queries = transpose_qkv(self.W_q(queries), self.num_heads)
        keys = transpose_qkv(self.W_k(keys), self.num_heads)
        values = transpose_qkv(self.W_v(values), self.num_heads)

        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(
                valid_lens, repeats=self.num_heads, dim=0
            )

        # output: (batch_size * num_heads, no. of queries,
        # num_hiddens/num_heads)
        output = self.attention(queries, keys, values, valid_lens)

        # output_concat: (batch_size, no. of queries, num_hiddens)
        output_concat = transpose_output(output, self.num_heads)

        return self.W_o(output_concat)


def transpose_qkv(X, num_heads):
    """为了多注意力头的并行计算改变形状"""
    # 输入X: (batch_size, queries或k-v个数, num_hiddens)
    # --> (~, ~, num_heads, num_hiddens / num_heads)
    # reshape: 按内存顺序(行优先)重新填入
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)

    # --> (~, num_heads, queries或k-v个数, ~)
    # permute: 先遍历第0维，然后第2维，然后第1维，最后第3维
    X = X.permute(0, 2, 1, 3)

    # --> (batch_size * num_heads, ~, ~)
    return X.reshape(-1, X.shape[2], X.shape[3])


def transpose_output(X, num_heads):
    """transpose_qkv的逆操作"""
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)
    




In [40]:
num_hiddens, num_heads = 100, 5
attention = MultiHeadAttention(num_hiddens, num_hiddens, num_hiddens,
                               num_hiddens, num_heads, 0.5)
attention.eval()

MultiHeadAttention(
  (attention): DotProductAttention(
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (W_q): Linear(in_features=100, out_features=100, bias=False)
  (W_k): Linear(in_features=100, out_features=100, bias=False)
  (W_v): Linear(in_features=100, out_features=100, bias=False)
  (W_o): Linear(in_features=100, out_features=100, bias=False)
)

In [41]:
batch_size, num_queries = 2, 4
num_kvpairs, valid_lens =  6, torch.tensor([3, 2])
X = torch.ones((batch_size, num_queries, num_hiddens))
Y = torch.ones((batch_size, num_kvpairs, num_hiddens))
attention(X, Y, Y, valid_lens).shape

torch.Size([2, 4, 100])